# YOLOv11n Action Classifier Training
Train basketball action classifier on Colab T4 GPU.

**Before running:** Upload the `action_classifier_yolo` folder to your Google Drive root (drag & drop the whole folder).

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2: Copy dataset from Drive to Colab local disk (faster training I/O)
import shutil
from pathlib import Path

src = Path('/content/drive/MyDrive/action_classifier_yolo')
dst = Path('/content/action_classifier_yolo')

if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(src, dst)
print(f'Dataset copied to {dst}')
print(f'Train images: {len(list((dst/"train/images").glob("*.jpg")))}')
print(f'Val images:   {len(list((dst/"valid/images").glob("*.jpg")))}')

In [ ]:
# Step 3: Write data.yaml with correct Colab paths
from pathlib import Path

dataset_root = Path('/content/action_classifier_yolo')

yaml_content = f"""path: {dataset_root}
train: train/images
val:   valid/images
test:  test/images

nc: 12
names: ['players', 'ball', 'number', 'player', 'player-dribble', 'player-fall', 'player-jump-shot', 'player-layup', 'player-screen', 'player-shot-block', 'referee', 'rim']
"""

(dataset_root / 'data.yaml').write_text(yaml_content)
print('data.yaml written:')
print(yaml_content)

In [ ]:
# Step 4: Install ultralytics
!pip install ultralytics -q

In [ ]:
# Step 5: Train
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

results = model.train(
    data='/content/action_classifier_yolo/data.yaml',
    epochs=80,
    imgsz=640,
    batch=16,
    device='cuda',
    project='/content/runs',
    name='action_classifier',
    exist_ok=True,
    patience=20,
    save_period=10,
    verbose=True,
)

print('Training complete!')

In [ ]:
# Step 6: Save best.pt to Google Drive
import shutil

src = '/content/runs/action_classifier/weights/best.pt'
dst = '/content/drive/MyDrive/action_classifier_best.pt'
shutil.copy2(src, dst)
print(f'best.pt saved to Google Drive → {dst}')
print('Download it and replace: runs/detect/models/action_classifier/weights/best.pt')

In [ ]:
# Optional: View training curves
from IPython.display import Image
Image('/content/runs/action_classifier/results.png')